# Synaptic Plasticity and Learning (CCNSS2026)

## Tutorial 2: Error-Gated Hebbian Rule: blind source separation

*Reference: Isomura, T. and Toyoizumi, T. (2018) “Error-Gated Hebbian Rule: A Local Learning Rule for Principal and Independent Component Analysis,” Scientific Reports, 8(1), p. 1835. Available at: https://doi.org/10.1038/s41598-018-20082-0.*

A *local* synaptic rule can perform Independent Component Analysis (ICA). 

Given independent source signals $\mathbf s$ and a mixing matrix $A$, the observed signals are $\mathbf x = A\mathbf s$. The goal is to learn a demixing matrix $W$ such that the outputs $\mathbf u = W\mathbf x$ are independent.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams["figure.dpi"] = 110


## Sources, mixing, and whitening

Three independent **sub-Gaussian** sources (sine, sawtooth, uniform noise), linearly mixed, then **whitened** (a decorrelation step the retina/LGN performs).

We build the classic **instantaneous linear mixture** model of blind source separation in three steps:

1. **Independent sources.** Generate $\mathbf s(t) = (s_1(t), s_2(t), s_3(t))^\top$ — a sine wave, a sawtooth, and uniform noise — then standardize each to zero mean, unit variance. All three have negative excess kurtosis,
$$\kappa_i = \frac{\langle s_i^4\rangle}{\langle s_i^2\rangle^2} - 3 < 0,$$
i.e. they are **sub-Gaussian** (flatter-than-Gaussian), which is what the EGHR prior below assumes.

2. **Linear mixing.** Stack the sources as rows of $S\in\mathbb{R}^{3\times T}$ and mix with a random matrix $A\in\mathbb{R}^{3\times 3}$ (entries i.i.d. $\mathcal N(0,1)$), giving the observed signals
$$X = A\,S, \qquad \mathbf x(t) = A\,\mathbf s(t).$$
$X$ plays the role of the mixed microphone/electrode recordings; $A$ is unknown to the learner.

3. **Centering + ZCA whitening.** Subtract the mean, $X_c = X - \langle X\rangle$, then remove all second-order (pairwise) correlations. With covariance $\Sigma = \operatorname{Cov}(X_c) = U D U^\top$ (eigendecomposition), the **ZCA whitening matrix** is $\Sigma^{-1/2} = U D^{-1/2}U^\top$, giving
$$Z = \Sigma^{-1/2} X_c, \qquad \operatorname{Cov}(Z) = I.$$
Whitening only fixes second-order statistics, so any remaining mixing between the whitened channels $Z$ must be an **orthogonal rotation** $W$ — recovering $\mathbf u = W\mathbf z \approx \mathbf s$ (up to permutation/sign) requires using higher-order (non-Gaussian) statistics, which is exactly what ICA — and the EGHR rule below — does.

In [ ]:
T = 1000
t = np.linspace(0, 1, T)
S = np.vstack([
    np.sin(2*np.pi*7*t),            # sine     (kurtosis ~ -1.5)
    2*((t*11) % 1) - 1,             # sawtooth (sub-Gaussian)
    rng.uniform(-1, 1, T),          # uniform  (sub-Gaussian)
])
S = (S - S.mean(1, keepdims=True)) / S.std(1, keepdims=True)
print("source kurtoses:", np.round((S**4).mean(1) - 3, 2))

A = rng.standard_normal((3, 3))     # random mixing
X = A @ S

Xc = X - X.mean(1, keepdims=True)   # center + ZCA-whiten
d, U = np.linalg.eigh(np.cov(Xc))
Z = (U @ np.diag(1/np.sqrt(d)) @ U.T) @ Xc


Visualize the sources and mixing in the following figure:

In [ ]:
def show(M, title, n=1000):
    fig, ax = plt.subplots(3, 1, figsize=(8, 3), sharex=True)
    for i in range(3):
        ax[i].plot(M[i, :n], lw=1)
        # ax[i].set_yticks([])
    ax[0].set_title(title); plt.tight_layout(); plt.show()

show(S, "Original independent sources")
show(X, "Observed mixtures (what the network receives)")


## The EGHR learning rule

The **EGHR-$\beta$** update (ICA case $\beta=0$) is

$$\dot W \;\propto\; -\big\langle\,\underbrace{(E(\mathbf u)-E_0)}_{\text{global error gate}}\;\underbrace{g(\mathbf u)\,\mathbf x^\top}_{\text{Hebbian}}\,\big\rangle,
\qquad E(\mathbf u)=-\log p_0(\mathbf u),\quad g=\partial E / \partial \mathbf{u}.$$

The bracket factorizes into a **scalar** $(E(\mathbf u)-E_0)$ — a *global* error signal broadcast
to every synapse (a **third factor**, like a neuromodulator) — times a **Hebbian** outer product
$g(u_i)\,x_j$ (post-nonlinearity × presynaptic input). No weight transport, no $W^\top$: every
quantity is locally available. 
With a **sub-Gaussian (uniform) prior**, 
$$p_0\propto e^{-u^4/4},$$
so 
$$g(u)=u^3\quad\text{and}\quad E(\mathbf u)=\tfrac14\sum_i u_i^4.$$


In [ ]:
# EXERCISE: implement the EGHR update rule for ICA. Use a sub-Gaussian prior (e.g. uniform) and track the correlation of the learned sources with the true sources.
g   = lambda u: u**3                      # score function for sub-Gaussian prior
Efn = lambda u: 0.25 * (u**4).sum(0)      # E(u) = sum_i u_i^4 / 4   (per-sample scalar)

def corr_recovery(W, Z, S):
    Uo = W @ Z
    M = np.abs(np.corrcoef(Uo, S)[:3, 3:]).copy()
    best = []
    for _ in range(3):                    # greedy best permutation match
        i, j = np.unravel_index(np.argmax(M), M.shape)
        best.append(M[i, j]); M[i, :] = -1; M[:, j] = -1
    return np.array(best)

W = np.linalg.qr(rng.standard_normal((3, 3)))[0]   # start from a random rotation
lr, batch, n_epoch = 0.02, 200, 40
Ebar, hist = 1.0, []
for ep in range(n_epoch):
    for b in range(0, T, batch):
        idx = rng.integers(0, T, batch)
        z = Z[:, idx]
        # YOUR CODE HERE
        # compute u = W @ z, the energy E = Efn(u), update the running estimate
        # Ebar (e.g. Ebar = 0.99*Ebar + 0.01*E.mean()), form the gating error term
        # gate = (E - Ebar), compute the gated Hebbian update
        # dW = -((gate * g(u)) @ z.T) / batch, apply it as W = W + lr * dW, then
        # renormalize each row of W to unit norm
        raise NotImplementedError("implement the EGHR update rule")
    hist.append(corr_recovery(W, Z, S).mean())

rec = corr_recovery(W, Z, S)
print("per-source |corr| with truth:", np.round(rec, 3), " mean:", round(rec.mean(), 3))


visualize the recovered source in the following figure:

In [ ]:
U_out = W @ Z
show(U_out, "Recovered sources  u = W z  (after EGHR)")

fig, ax = plt.subplots(1, 1, figsize=(5, 3))
ax.plot(hist, marker="o", ms=3)
ax.set_xlabel("epoch"); ax.set_ylabel("mean |corr| with true sources")
ax.set_title("EGHR convergence"); ax.set_ylim(0, 1.05); ax.axhline(1, ls="--", c="gray")
plt.tight_layout(); plt.show()


## Homework questions:
1. Implement the EGHR algorithm with mixed ICA and PCA objectives ($\beta>0$). How does the learned $W$ change as you vary $\beta$?
    $$L \equiv (1-\beta)\underbrace{\left\langle \frac{1}{2}\left(E(\mathbf{u}) - \langle E(\mathbf{u})\rangle\right)^2 - E(\mathbf{u})\right\rangle}_{\text{ICA term}} + \beta\underbrace{\left\langle \frac{1}{2}\left(E_u(\mathbf{u}) - E_x(\mathbf{x})\right)^2\right\rangle}_{\text{PCA term}},$$
    where
    $$\begin{aligned} E(\mathbf{u}) &\equiv -\log p_0(\mathbf{u}), \\[6pt] E_u(\mathbf{u}) &\equiv \frac{1}{2}\left(|\mathbf{u}|^2 - \langle |\mathbf{u}|^2 \rangle\right), \\[6pt] E_x(\mathbf{x}) &\equiv \frac{1}{2}\left(|\mathbf{x}|^2 - \langle |\mathbf{x}|^2 \rangle\right). \end{aligned}.$$
    It gives us a **continuum of objectives** between ICA and PCA,
    $$\dot{W} \propto -\frac{\partial L}{\partial W} = -\left\langle (1-\beta)\underbrace{(E(\mathbf{u}) - E_0)g(\mathbf{u})\mathbf{x}^T}_{\text{ICA term}} + \beta\underbrace{(E_u(\mathbf{u}) - E_x(\mathbf{x}))\mathbf{u}\mathbf{x}^T}_{\text{PCA term}}\right\rangle,$$
    with $\beta=0$ giving pure ICA and $\beta=1$ giving pure PCA.

2. Try separate super-Gaussian signals (e.g. Laplace, exponential) instead of sub-Gaussian signals. How does the EGHR rule need to be modified? What happens if you use the wrong prior?

## Takeaways

- **Local ICA.** The EGHR separates independent sources using only signals available *at the
  synapse*: a presynaptic input, a postsynaptic nonlinearity, and **one global scalar** error.
- That global scalar is the **third factor** — exactly the motif that generalizes Hebb to a
  learning rule (cf. neuromodulators, reward, error). Demos 3–4 follow this same third factor into
  reinforcement learning, where it becomes the **dopamine reward-prediction error** (the TD error
  $\delta$).
- **Connection to L1:** with $\beta=1$ the same rule reduces to Oja-style PCA (Demo 1). One
  family, two classic computations — chosen by the objective.
